## This is the code to train the model and acquire influence for Neighbour Influence 2 Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_blobs function. The default version code provides the synthetic dataset with 8000 samples and 10 features in total. The dataset has only two labels, so it is a binary classification problem. **Within the 8000 samples, 4000 samples are assigned to class 0 and another 4000 samples are assigned to class 1. Within class 1, we further split them into 2000 samples each. One group has a larger density, while the other has a larger neighbour distance within the group.** We name them respectively, group 0, group 1 dense and group 1 sparse. Then, 500 samples are chosen to be the test set. The distribution of test sets within each group is (250, 125, 125). Then the remaining 7500 samples are the train set. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. Also, a mapping file recording the sample id, label and cluster ID will be saved. The result lists could then be fed into other analyses.

**By default, the biggest difference is on how to construct each cluster group. For all the other common information, please refer to the base code for more detailed explanation.**

**Guideline**:  
Read in / Construct Datasets -> **Specify the standard deviation of each cluster group** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **It could be fine to stop here and feed the results into the analysis code.** -> Or either Change the standard deviation of each cluster group / change the seed value and repeat all the process (**This is for multi-verification**) -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [153]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [154]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [155]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [156]:
import random
from keras.optimizers import SGD

In [157]:
from sklearn.datasets import make_classification
from sklearn.datasets import make_blobs

In [158]:
import seaborn as sns
import matplotlib.pyplot as plt

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 8000 samples, 10 features with binary classification problems. The 8000 samples are constructed from 4000 class 0 samples, 2000 class 1 dense samples and 2000 class 1 sparse samples. The later options will turn that into a 10 features, 7500 train set and 500 test set sample. The test set has a distribution of 250 class 0 samples, 125 class 1 dense samples and 125 class 1 sparse samples. Both sets will then be turned into TensorFlow format and will wait for training.

**The most important thing in this code is changing the standard deviation of each cluster. In this experiment, all the other things are fixed, but the standard deviation is changing to test how the density in each group affects its influence.**

1. Set your default setting here. Total size represent the Total Dataset Size this time. train_sizes = total_size - test_size. Sep to make sure the dataset is distinguishable. **The most important thing here is to specify the standard deviation for each cluster group. In most cases, the 0 group shall have same standard value as the 1 sparse group, and the 1 dense group shall have a smaller std value compared with them.**

In [159]:
total_size = 8500
test_size = 500
n_features=10
seed=42       
std1_dense = 0.1   
std1_sparse = 1.0

2. To make sure the two cluster is having different density, we use make_blobs this time. We set the cluster density by changing the standard deviation of the cluster. Each cluster has the same number of samples, only differnce in density. Each class has same number of samples. The test set contains equal number across class and equal number across cluster. Say we have 8000 samples in total, then 500 shall be the test set. In the whole 8000 pool, class 0 accounts for 4000 samples, class 1 sparse for 2000 and class 1 dense for 2000. In the test set, 250 will come from group 0, 125 from 1 dense and 125 from 1 sparse. 

In [160]:
n_per_class = total_size // 2

In [161]:
big_size = n_per_class // 2
small_size = n_per_class - big_size

In [162]:
c0 = np.zeros(n_features)
c0[0] = -sep

c1 = np.zeros(n_features)
c1[0] = sep

In [163]:
X0_dense, _ = make_blobs(
    n_samples=small_size,
    centers=[c0],
    cluster_std=std1_dense,
    n_features=n_features,
    random_state=seed
)

X0_sparse, _ = make_blobs(
    n_samples=big_size,
    centers=[c0],
    cluster_std=std1_sparse,
    n_features=n_features,
    random_state=seed + 1
)

X1_dense, _ = make_blobs(
    n_samples=small_size,
    centers=[c1],
    cluster_std=std1_dense,
    n_features=n_features,
    random_state=seed + 2
)

X1_sparse, _ = make_blobs(
    n_samples=big_size,
    centers=[c1],
    cluster_std=std1_sparse,
    n_features=n_features,
    random_state=seed + 3
)

In [164]:
X = np.vstack([X0_dense, X0_sparse, X1_dense, X1_sparse])
y = np.hstack([
    np.zeros(big_size, dtype=int),
    np.zeros(small_size, dtype=int),
    np.ones(big_size, dtype=int),
    np.ones(small_size, dtype=int),
])

In [165]:
cluster_id = (
    (["0_dense"] * big_size) +
    (["0_sparse"] * small_size) +
    (["1_dense"] * big_size) +
    (["1_sparse"] * small_size)
)
rng = np.random.RandomState(seed)
idx = rng.permutation(X.shape[0])
X = X[idx]; y = y[idx]
cluster_id = np.array(cluster_id, dtype=object)[idx]

In [166]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(n_features)])
df['label'] = y
df['id'] = np.arange(1, len(df) + 1)
df["cluster_id"] = cluster_id 
print(df)
print(df["label"].value_counts())

      feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0     -1.544001  -0.078971   0.055722   0.075512  -0.046838   0.185928   
1     -2.127460  -1.713432   1.761928   0.459091   1.295558   0.440680   
2      1.382018  -0.008870  -0.032759  -0.028917   0.050245  -0.011609   
3     -3.729756  -0.076239   0.202808  -1.252742   1.133601   0.960555   
4      1.452696   0.112198  -0.031943   0.022873   0.128068   0.025062   
...         ...        ...        ...        ...        ...        ...   
8495  -1.420299   0.027048   0.098044   0.136673  -0.145387  -0.075402   
8496  -2.555474  -1.231249  -0.394289  -0.687417   0.244899  -1.416973   
8497  -2.907790  -0.437314  -0.337486   0.383160   0.501470   0.964410   
8498   1.881569  -0.258261   0.012270  -1.120603  -0.151361  -2.262063   
8499   1.437868  -1.352858   1.556934   0.268078   0.324077   0.915148   

      feature_7  feature_8  feature_9  feature_10  label    id cluster_id  
0      0.009018  -0.169543   0.0359

In [167]:
print(df["cluster_id"].value_counts())

cluster_id
0_dense     2125
0_sparse    2125
1_dense     2125
1_sparse    2125
Name: count, dtype: int64


3. We also store the label, id and cluster id in a dataframe so that we could know which cluster each sample is from.

In [168]:
id_label_df = df[["id", "label","cluster_id"]].copy()
print(id_label_df)
id_label_df.to_csv("label_sp1_de01_newset.csv",index = False)

        id  label cluster_id
0        1      0    0_dense
1        2      0   0_sparse
2        3      1    1_dense
3        4      0   0_sparse
4        5      1    1_dense
...    ...    ...        ...
8495  8496      0    0_dense
8496  8497      0   0_sparse
8497  8498      0   0_sparse
8498  8499      1   1_sparse
8499  8500      1   1_sparse

[8500 rows x 3 columns]


In [169]:
n0 = test_size //2
n1_big = (test_size - n0) // 2
n1_small = test_size - n0 - n1_big
n0,n1_big,n1_small

(250, 125, 125)

In [170]:
g = df.groupby("cluster_id", group_keys=False)
test_df = pd.concat([
    g.get_group("0_dense").sample(n=n1_big, random_state=seed, replace=False),
    g.get_group("0_sparse").sample(n=n1_small, random_state=seed, replace=False),
    g.get_group("1_dense").sample(n=n1_big, random_state=seed, replace=False),
    g.get_group("1_sparse").sample(n=n1_small, random_state=seed, replace=False)
]).sample(frac=1, random_state=seed)

train_df = df.drop(test_df.index).reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

In [171]:
print(test_df.groupby("label").get_group(0))
print(test_df["cluster_id"].value_counts())

     feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0     0.313056  -0.537659  -0.924056  -0.199434  -1.434735  -0.317748   
1    -2.426371   0.849173  -0.991331   0.851878   0.429450   1.280641   
2    -1.441257   0.002735   0.149272  -0.137444   0.038682   0.007981   
3    -3.940531   0.001153   0.290079  -0.083902  -0.896200   0.696436   
4    -1.000351  -0.057762  -1.766418  -0.243402  -0.565577   1.955108   
..         ...        ...        ...        ...        ...        ...   
487  -0.877987  -0.819606   1.933939   0.989296   0.800737  -1.711942   
492  -2.549069  -1.629814   0.822077   0.532704  -0.395127   0.223475   
493  -0.665176   1.672851  -0.297259   0.637847  -0.812225  -1.114489   
497  -1.362648   0.015685   0.152139  -0.064953  -0.133924   0.171676   
499  -1.839421   1.231649  -0.549731   0.609219  -0.500246  -0.909141   

     feature_7  feature_8  feature_9  feature_10  label    id cluster_id  
0    -0.368605  -0.601571  -1.757704   -1.084952

In [172]:
print(train_df["cluster_id"].value_counts())

cluster_id
0_dense     2000
0_sparse    2000
1_dense     2000
1_sparse    2000
Name: count, dtype: int64


In [173]:
X_train = train_df.drop(columns=["label","cluster_id"])
y_train = train_df["label"]
IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
X_train = np.hstack((X_train, IDs))
y_train = to_categorical(y_train.values,num_classes=2)

print(X_train)

[[-1.5440007e+00 -7.8971125e-02  5.5722207e-02 ...  3.5959888e-02
  -1.6821226e-01  1.0000000e-10]
 [-2.1274598e+00 -1.7134318e+00  1.7619280e+00 ...  4.1891301e-01
  -7.9682171e-01  2.0000000e-10]
 [ 1.3820181e+00 -8.8695800e-03 -3.2758545e-02 ...  3.0787544e-02
   1.1239191e-01  3.0000000e-10]
 ...
 [-2.9077904e+00 -4.3731382e-01 -3.3748639e-01 ...  9.3783826e-02
  -1.5681204e+00  8.4980002e-07]
 [ 1.8815688e+00 -2.5826097e-01  1.2269626e-02 ...  5.6100363e-01
   7.3347908e-01  8.4990000e-07]
 [ 1.4378681e+00 -1.3528582e+00  1.5569345e+00 ... -6.3488746e-01
  -7.5661951e-01  8.4999999e-07]]


In [174]:
X_test = test_df.drop(columns=["label","cluster_id"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test)

[[ 3.13056409e-01 -5.37658632e-01 -9.24056351e-01 ... -1.75770366e+00
  -1.08495176e+00  2.14200000e-07]
 [-2.42637086e+00  8.49172592e-01 -9.91330683e-01 ...  6.33728951e-02
   3.51799726e-01  3.49900006e-07]
 [-1.44125664e+00  2.73532257e-03  1.49271578e-01 ...  6.60667494e-02
   1.13109164e-01  5.53300026e-07]
 ...
 [-1.36264837e+00  1.56852920e-02  1.52138829e-01 ... -1.50660366e-01
   8.84041786e-02  2.40600002e-07]
 [ 1.51194501e+00 -1.48128103e-02  9.96995065e-03 ... -1.02610908e-01
   1.35148186e-02  2.49700008e-07]
 [-1.83942115e+00  1.23164904e+00 -5.49731374e-01 ... -2.46518478e-01
   8.93332064e-01  4.11900004e-07]]


4 (Optional) The following code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [175]:
# X_all = np.vstack([X_train, X_test])

In [176]:
# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

In [177]:
# from sklearn.metrics import pairwise_distances
# from sklearn.manifold import MDS

In [178]:
# D = pairwise_distances(X_all) 

In [179]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

5. Now we have the train_ds and test_ds for training

In [180]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 500 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

In [181]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [182]:
seed_value = 41
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 300
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
initial_model = tf.keras.models.clone_model(model)
initial_model.set_weights(model.get_weights())
model_list.append(InfluenceModel(initial_model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  checkpoint_model = tf.keras.models.clone_model(model)
  checkpoint_model.set_weights(model.get_weights())
  model_list.append(InfluenceModel(checkpoint_model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

32/32 - 1s - loss: 0.7479 - accuracy: 0.5340 - val_loss: 0.6406 - val_accuracy: 0.6660 - 672ms/epoch - 21ms/step
32/32 - 0s - loss: 0.5425 - accuracy: 0.7819 - val_loss: 0.4399 - val_accuracy: 0.9040 - 131ms/epoch - 4ms/step
32/32 - 0s - loss: 0.3652 - accuracy: 0.9091 - val_loss: 0.3021 - val_accuracy: 0.9220 - 133ms/epoch - 4ms/step
32/32 - 0s - loss: 0.2625 - accuracy: 0.9299 - val_loss: 0.2368 - val_accuracy: 0.9280 - 127ms/epoch - 4ms/step
32/32 - 0s - loss: 0.2115 - accuracy: 0.9399 - val_loss: 0.2027 - val_accuracy: 0.9420 - 141ms/epoch - 4ms/step
32/32 - 0s - loss: 0.1832 - accuracy: 0.9438 - val_loss: 0.1818 - val_accuracy: 0.9440 - 127ms/epoch - 4ms/step
32/32 - 0s - loss: 0.1652 - accuracy: 0.9467 - val_loss: 0.1678 - val_accuracy: 0.9460 - 128ms/epoch - 4ms/step
32/32 - 0s - loss: 0.1527 - accuracy: 0.9492 - val_loss: 0.1578 - val_accuracy: 0.9460 - 131ms/epoch - 4ms/step
32/32 - 0s - loss: 0.1435 - accuracy: 0.9509 - val_loss: 0.1502 - val_accuracy: 0.9520 - 122ms/epoch - 

In [183]:
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [184]:
train_logits = model.predict(
    X_train,
    batch_size=256,
    verbose=0
)

per_sample_loss_fn = tf.keras.losses.CategoricalCrossentropy(
    from_logits=True,
    reduction=tf.keras.losses.Reduction.NONE
)

train_losses = per_sample_loss_fn(
    y_train,
    train_logits
).numpy()

# Recover the original training IDs and labels
loss_df = pd.DataFrame({
    "Train_ID": train_ids,
    "Training_Loss": train_losses
})

In [185]:
loss_df.to_csv("loss_sp1_de01_newset.csv",index = False)

# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC.  
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [186]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [187]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

      Train_ID         Score
0            1  5.214134e-04
1            2  1.594855e-03
2            3  1.329427e-03
3            4  3.667717e-07
4            5  1.172189e-03
...        ...           ...
7995      8496  6.447583e-04
7996      8497  1.077284e-04
7997      8498  4.179900e-05
7998      8499  9.604474e-05
7999      8500  1.973069e-03

[8000 rows x 2 columns]


2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

In [188]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

      Train_ID     Score
0            1  0.000160
1            2  0.000181
2            3 -0.000023
3            4  0.000096
4            5 -0.000051
...        ...       ...
7995      8496  0.000119
7996      8497  0.000147
7997      8498  0.000211
7998      8499  0.000056
7999      8500 -0.000017

[8000 rows x 2 columns]


3. Here we turn both influence lists to the ranked influence lists and then store them for further processing.

In [189]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [190]:
TracIn_sorted.to_csv("TC_sp1_de01_newset.csv",index = False)
df_sorted.to_csv("IF_sp1_de01_newset.csv",index = False)